# Cloud-Native Data Engineering
## Assignment 1 Part B: End-to-End Ingestion Pipeline (Books to Scrape)

**Objective:** Build a Python-based data ingestion pipeline that scrapes a mock e-commerce bookstore (`books.toscrape.com`), structures the unstructured HTML into a Pandas DataFrame, cleans the data, and performs basic aggregations.

### Assignment Requirements:
1. **Web Scraping:** Use `requests` and `BeautifulSoup` to scrape book data (Title, Price, Star Rating, and Availability).
2. **Pagination Pipeline:** Write a loop to scrape the first 5 pages of the catalog.
3. **Pandas Fundamentals:** Load the scraped data into a DataFrame and clean the datatypes (e.g., convert string prices to floats).
4. **Aggregation:** Calculate summary statistics using Pandas.

In [1]:
# Import required libraries
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

---
### Part 1 & 2: Building the Web Scraping Pipeline with Pagination

We need to extract the following from each book:
- **Title:** The full name of the book.
- **Price:** The cost of the book.
- **Star Rating:** The text-based star rating (e.g., 'Three', 'One').
- **Availability:** Whether it is in stock.

In [2]:
#### Data engineers frequently need to write scripts that navigate through multiple pages of a website to extract records.
##### Your Task: Use the requests and BeautifulSoup libraries to scrape the first 5 pages of the catalog.
##### For each book, extract the Title, Price, Star Rating, and Availability.

def scrape_books_pipeline(num_pages=5):
    """
    Scrapes the specified number of pages from books.toscrape.com
    and returns a list of dictionaries containing book data.
    """
    base_url = "http://books.toscrape.com/catalogue/page-{}.html"
    books_data = []

    print(f"Starting scraping pipeline for {num_pages} pages...")

    for page in range(1, num_pages + 1):
        url = base_url.format(page)

        # TODO: Use the requests library to perform an HTTP GET request on the 'url'
        response = requests.get(url)

        # Check if the request was successful
        if response.status_code != 200:
            print(f"Failed to retrieve page {page}. Status code: {response.status_code}")
            continue

        # TODO: Parse the response text using BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser') 

        # Each book is contained within an <article class="product_pod">
        # TODO: Find all book containers (Hint: inspect the HTML for <article class="product_pod">)
        book_containers = soup.find_all('article', class_='product_pod')

        for book in book_containers:
            # Extract Title (found in the 'title' attribute of the <a> tag inside <h3>)
            # TODO: Extract the book's Title
            title = book.h3.a['title']

            # TODO: Extract the book's Price
            price = book.find('p', class_='price_color').get_text(strip=True)

            # TODO: Extract the Availability status (e.g., "In stock"; strip whitespace and newlines)
            availability = book.find('p', class_='availability').get_text(strip=True)

            # TODO: Extract Star Rating (It's the second class in the <p class="star-rating ..."> tag)
            star_rating = book.find('p', class_='star-rating')['class'][1]

            # Append to our list
            books_data.append({
                'Title': title,
                'Price': price,
                'Rating': star_rating,
                'Availability': availability
            })

        print(f"Successfully scraped page {page}")
        # Be polite to the server during scraping
        time.sleep(1)

    print(f"Pipeline complete. Extracted {len(books_data)} books.\n")
    return books_data

# Execute the pipeline
raw_books_list = scrape_books_pipeline(num_pages=5)

Starting scraping pipeline for 5 pages...
Successfully scraped page 1
Successfully scraped page 2
Successfully scraped page 3
Successfully scraped page 4
Successfully scraped page 5
Pipeline complete. Extracted 100 books.



---
### Part 3: Pandas Fundamentals - Data Loading and Cleaning

The data we scraped is currently in string format. To perform mathematical aggregations, we must clean the data (e.g., remove the '£' sign and convert prices to floats).
Your Task: Convert the raw_books_list into a Pandas DataFrame.

In [3]:
# 1. TODO: Load 'raw_books_list' into a Pandas DataFrame called 'df_books'
df_books = pd.DataFrame(raw_books_list)

print("--- Raw DataFrame Head ---")
display(df_books.head())

--- Raw DataFrame Head ---


,Title,Price,Rating,Availability
0,A Light in the Attic,Â£51.77,Three,In stock
1,Tipping the Velvet,Â£53.74,One,In stock
2,Soumission,Â£50.10,One,In stock
3,Sharp Objects,Â£47.82,Four,In stock
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock


In [4]:
# 3. Data Cleaning: Convert Price from string (e.g., '£51.77') to float
# Your Task: Remove the pound symbol (£) and convert the price column into a numeric float.
# Then, map the text-based star ratings to actual integers.

# TODO: Clean the 'Price' column. Remove the '£' symbol and convert the column to a float. Note ‘Â£'
# Save the result in a new column called 'Price_Float'.
df_books['Price_Float'] = df_books["Price"].str.replace("Â£", "", regex=False).astype(float)

# 3. Data Cleaning: Map text-based ratings to numeric integers
# TODO: Create a dictionary mapping text ratings ('One', 'Two', etc.) to integers (1, 2, etc.)
rating_mapping = {
     "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# TODO: Apply the mapping to the 'Rating' column and save it as 'Rating_Num'
df_books['Rating_Num'] = df_books["Rating"].map(rating_mapping)

print("\n--- Cleaned DataFrame DataTypes ---")
print(df_books.dtypes)

print("\n--- Cleaned DataFrame Head ---")
display(df_books[['Title', 'Price_Float', 'Rating_Num', 'Availability']].head())


--- Cleaned DataFrame DataTypes ---
Title               str
Price               str
Rating              str
Availability        str
Price_Float     float64
Rating_Num        int64
dtype: object

--- Cleaned DataFrame Head ---


,Title,Price_Float,Rating_Num,Availability
0,A Light in the Attic,51.77,3,In stock
1,Tipping the Velvet,53.74,1,In stock
2,Soumission,50.10,1,In stock
3,Sharp Objects,47.82,4,In stock
4,Sapiens: A Brief History of Humankind,54.23,5,In stock


---
### Part 4: Data Manipulation and Aggregation

Now that our data is clean and structured, we can answer business questions using Pandas aggregations.

Your Task: Calculate the overall average price of a book, and then group the data to find the average price based on the star rating

In [5]:
# Question 1: What is the average price of a book in our scraped dataset?
# TODO: Calculate the mean of the 'Price_Float' column
avg_price = df_books['Price_Float'].mean()
print(f"Average Book Price: £{avg_price:.2f}")

# Question 2: What is the distribution of star ratings? (e.g., How many 5-star books vs 1-star books?)
# TODO: Use value_counts() on the 'Rating_Num' column
print("\n--- Count of Books by Star Rating ---")
rating_counts =  df_books['Rating_Num'].value_counts()
display(rating_counts)

# Question 3: What is the average price of books grouped by their Star Rating?
# TODO: Group the dataframe by 'Rating_Num' and calculate the mean of 'Price_Float'
print("\n--- Average Price by Star Rating ---")
price_by_rating = df_books.groupby('Rating_Num')[['Price_Float']].mean()
price_by_rating.rename(columns={'Price_Float': 'Avg_Price'}, inplace=True)
display(price_by_rating.round(2))

# Final Step: Save the cleaned dataset to a CSV called 'books_dataset_silver.csv' (Simulating Bronze/Silver layer storage)
# TODO: Export the dataframe to CSV without the index column
df_books.to_csv('books_dataset_silver.csv', index=False)

print("\nPipeline Complete: Data saved to 'books_dataset_silver.csv'")

Average Book Price: £34.56

--- Count of Books by Star Rating ---


Rating_Num
3    22
1    22
5    19
2    19
4    18
Name: count, dtype: int64


--- Average Price by Star Rating ---


,Avg_Price
Rating_Num,
1,35.52
2,35.91
3,36.84
4,33.98
5,30.01



Pipeline Complete: Data saved to 'books_dataset_silver.csv'
